In [ ]:
!pip install -q typer
!pip install -q docling
!pip install -q docling_core
!pip install -q langchain_text_splitters
!pip install -q filetype
!pip install -q docling-ibm-models
!pip install -q docling-parse

!pip install -q \
    pydantic \
    rapidocr \
    huggingface_hub \
    pillow \
    pypdfium2 \
    requests \
    pylatexenc

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 6.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.7/42.7 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.7/62.7 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.8/472.8 kB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 73.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 635.4/635.4 kB 36.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.1/98.1 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.7/99.7 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.1/108.1 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.0/345.0 kB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 14.8 MB/s eta 0:00:00
   ━

In [ ]:
# ============================================================
# IMPORTS
# ============================================================

import re
import json
from docling.document_converter import DocumentConverter
from langchain_text_splitters import MarkdownHeaderTextSplitter


# ============================================================
# CONFIG
# ============================================================

SOURCE_URL = "https://www.iit.edu/sites/default/files/2026-02/2025-2026%20Student%20Handbook%20Final%20Copy_0.pdf"

DOC_TYPE = "pdf"
DOC_NAME = "student handbook.pdf"
DOC_ID = "Student_handbook"

OUTPUT_FILE = "chunks.json"

MAX_TOKENS = 350
OVERLAP_RATIO = 0.20


# ============================================================
# CLEAN TEXT FUNCTION
# ============================================================

def clean_chunk_text(text: str) -> str:

    if not text:
        return ""

    text = re.sub(r'<!--.*?-->', ' ', text)
    text = re.sub(r'!\[.*?\]\(.*?\)', ' ', text)

    text = re.sub(r'^\|\s*[-:]+\s*(\|\s*[-:]+\s*)+\|?\s*$', ' ', text, flags=re.MULTILINE)

    def convert_table_row(match):
        row = match.group(0)
        cells = [c.strip() for c in row.strip('|').split('|')]
        return " - ".join(cells)

    text = re.sub(r'^\|.*\|$', convert_table_row, text, flags=re.MULTILINE)

    text = re.sub(r'\.{3,}', ' ', text)
    text = re.sub(r'-{3,}', ' ', text)

    text = text.replace("|", " ")

    text = re.sub(r'[•▪◦‣]', ' ', text)
    text = re.sub(r'[^\x00-\x7F]+', ' ', text)

    text = re.sub(r'(?<!\n)\n(?!\n)', ' ', text)

    text = re.sub(r'\s+', ' ', text)

    return text.strip()


# ============================================================
# PROMOTE CAPS HEADINGS
# ============================================================

def promote_caps_to_headers(markdown: str) -> str:

    lines = markdown.split("\n")
    new_lines = []

    for line in lines:

        stripped = line.strip()

        if (
            len(stripped) > 4
            and len(stripped) < 120
            and stripped.upper() == stripped
            and any(c.isalpha() for c in stripped)
            and not stripped.startswith("#")
        ):
            new_lines.append(f"## {stripped}")
        else:
            new_lines.append(line)

    return "\n".join(new_lines)


# ============================================================
# TOKEN COUNT
# ============================================================

def token_count(text):
    return len(text.split())


# ============================================================
# CHUNK ID
# ============================================================

def generate_chunk_id(index):
    return f"{DOC_ID}_{index}"


# ============================================================
# SENTENCE SPLIT
# ============================================================

def split_sentences(text):

    sentences = re.split(r'(?<=[.!?])\s+', text)

    return [s.strip() for s in sentences if s.strip()]


# ============================================================
# RECHUNK FUNCTION
# ============================================================

def rechunk_text(text):

    sentences = split_sentences(text)

    chunks = []
    current = []
    tokens = 0

    for s in sentences:

        t = token_count(s)

        if tokens + t > MAX_TOKENS:

            chunk = " ".join(current)

            chunks.append(chunk)

            overlap = int(len(current) * OVERLAP_RATIO)

            current = current[-overlap:]

            tokens = token_count(" ".join(current))

        current.append(s)
        tokens += t

    if current:
        chunks.append(" ".join(current))

    return chunks


# ============================================================
# BUILD PAGE MAP FROM DOCLING
# ============================================================

def build_page_map(document):

    page_map = []

    for item, _ in document.iterate_items():

        if hasattr(item, "text") and item.text:

            page = None

            if item.prov:
                page = item.prov[0].page_no

            clean_text = clean_chunk_text(item.text)

            if clean_text:

                page_map.append({
                    "text": clean_text,
                    "page": page
                })

    return page_map


# ============================================================
# FIND PAGE RANGE
# ============================================================

def find_page_range(chunk_text, page_map):

    pages = set()

    chunk_lower = chunk_text.lower()

    for entry in page_map:

        entry_text = entry["text"].lower()

        if entry_text and entry_text in chunk_lower:
            pages.add(entry["page"])

    if not pages:
        return None, None

    return min(pages), max(pages)


# ============================================================
# EXTRACT MARKDOWN USING DOCLING
# ============================================================

def extract_markdown(source):

    converter = DocumentConverter()

    result = converter.convert(source)

    document = result.document

    markdown = document.export_to_markdown()

    return document, markdown


# ============================================================
# LANGCHAIN SPLIT
# ============================================================

def split_markdown(markdown):

    headers = [
        ("#", "topic"),
        ("##", "section"),
        ("###", "subsection"),
    ]

    splitter = MarkdownHeaderTextSplitter(
        headers_to_split_on=headers
    )

    docs = splitter.split_text(markdown)

    return docs


# ============================================================
# BUILD FINAL CHUNKS
# ============================================================

def build_chunks(docs, page_map):

    chunks = []
    index = 1

    for doc in docs:

        content = clean_chunk_text(doc.page_content)

        if not content or len(content) < 20:
            continue

        meta = doc.metadata

        topic = meta.get("topic")
        section = meta.get("section")
        subsection = meta.get("subsection")

        final_topic = subsection or section or topic or "General"

        pieces = rechunk_text(content)

        for piece in pieces:

            chunk = {

                "chunk_id": generate_chunk_id(index),
                "doc_type": DOC_TYPE,
                "doc_name": DOC_NAME,
                "source_url": SOURCE_URL,
                "Topic": final_topic,
                "num_tokens": token_count(piece),
                "content": piece

            }

            chunks.append(chunk)
            index += 1

    return chunks


# ============================================================
# MAIN PIPELINE
# ============================================================

def process():

    print("Extracting with Docling...")

    document, markdown = extract_markdown(SOURCE_URL)

    print("Promoting headings...")

    markdown = promote_caps_to_headers(markdown)

    print("Building page map...")

    page_map = build_page_map(document)

    print("Splitting document...")

    docs = split_markdown(markdown)

    print("Building chunks...")

    chunks = build_chunks(docs, page_map)

    print("Saving...")

    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        json.dump(chunks, f, indent=2, ensure_ascii=False)

    print(f"SUCCESS: {len(chunks)} chunks created")


# ============================================================
# RUN
# ============================================================

process()

Extracting with Docling...


[INFO] 2026-03-08 14:03:17,572 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-03-08 14:03:17,574 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-03-08 14:03:17,745 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_infer.pth
[INFO] 2026-03-08 14:03:17,747 [RapidOCR] main.py:50: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_infer.pth
[INFO] 2026-03-08 14:03:18,154 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-03-08 14:03:18,156 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-03-08 14:03:18,179 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_ptocr_mobile_v2.0_cls_infer.pth
[INFO] 2026-03-08 14:03:18,182 [RapidOCR] main.py:50: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_ptocr_mobile_v2.0_cls_infer.pth
[INFO] 2026-03-08 14:03:18,349 [RapidO

Promoting headings...
Building page map...
Splitting document...
Building chunks...
Saving...
SUCCESS: 374 chunks created


In [ ]:
# ============================================================
# IMPORTS
# ============================================================

import re
import json
from docling.document_converter import DocumentConverter
from langchain_text_splitters import MarkdownHeaderTextSplitter


# ============================================================
# CONFIG
# ============================================================

SOURCE_URL = "https://webmaster.iit.edu/files/graduate-academic-affairs/co-terminal-student-handbook.pdf"

DOC_TYPE = "pdf"
DOC_NAME = "Coterminal_student handbook.pdf"
DOC_ID = "Coterminal_Student_handbook"

OUTPUT_FILE = "coterminal-chunks.json"

MAX_TOKENS = 350
OVERLAP_RATIO = 0.20


# ============================================================
# CLEAN TEXT FUNCTION
# ============================================================

def clean_chunk_text(text: str) -> str:

    if not text:
        return ""

    text = re.sub(r'<!--.*?-->', ' ', text)
    text = re.sub(r'!\[.*?\]\(.*?\)', ' ', text)

    text = re.sub(r'^\|\s*[-:]+\s*(\|\s*[-:]+\s*)+\|?\s*$', ' ', text, flags=re.MULTILINE)

    def convert_table_row(match):
        row = match.group(0)
        cells = [c.strip() for c in row.strip('|').split('|')]
        return " - ".join(cells)

    text = re.sub(r'^\|.*\|$', convert_table_row, text, flags=re.MULTILINE)

    text = re.sub(r'\.{3,}', ' ', text)
    text = re.sub(r'-{3,}', ' ', text)

    text = text.replace("|", " ")

    text = re.sub(r'[•▪◦‣]', ' ', text)
    text = re.sub(r'[^\x00-\x7F]+', ' ', text)

    text = re.sub(r'(?<!\n)\n(?!\n)', ' ', text)

    text = re.sub(r'\s+', ' ', text)

    return text.strip()


# ============================================================
# PROMOTE CAPS HEADINGS
# ============================================================

def promote_caps_to_headers(markdown: str) -> str:

    lines = markdown.split("\n")
    new_lines = []

    for line in lines:

        stripped = line.strip()

        if (
            len(stripped) > 4
            and len(stripped) < 120
            and stripped.upper() == stripped
            and any(c.isalpha() for c in stripped)
            and not stripped.startswith("#")
        ):
            new_lines.append(f"## {stripped}")
        else:
            new_lines.append(line)

    return "\n".join(new_lines)


# ============================================================
# TOKEN COUNT
# ============================================================

def token_count(text):
    return len(text.split())


# ============================================================
# CHUNK ID
# ============================================================

def generate_chunk_id(index):
    return f"{DOC_ID}_{index}"


# ============================================================
# SENTENCE SPLIT
# ============================================================

def split_sentences(text):

    sentences = re.split(r'(?<=[.!?])\s+', text)

    return [s.strip() for s in sentences if s.strip()]


# ============================================================
# RECHUNK FUNCTION
# ============================================================

def rechunk_text(text):

    sentences = split_sentences(text)

    chunks = []
    current = []
    tokens = 0

    for s in sentences:

        t = token_count(s)

        if tokens + t > MAX_TOKENS:

            chunk = " ".join(current)

            chunks.append(chunk)

            overlap = int(len(current) * OVERLAP_RATIO)

            current = current[-overlap:]

            tokens = token_count(" ".join(current))

        current.append(s)
        tokens += t

    if current:
        chunks.append(" ".join(current))

    return chunks


# ============================================================
# BUILD PAGE MAP FROM DOCLING
# ============================================================

def build_page_map(document):

    page_map = []

    for item, _ in document.iterate_items():

        if hasattr(item, "text") and item.text:

            page = None

            if item.prov:
                page = item.prov[0].page_no

            clean_text = clean_chunk_text(item.text)

            if clean_text:

                page_map.append({
                    "text": clean_text,
                    "page": page
                })

    return page_map


# ============================================================
# FIND PAGE RANGE
# ============================================================

def find_page_range(chunk_text, page_map):

    pages = set()

    chunk_lower = chunk_text.lower()

    for entry in page_map:

        entry_text = entry["text"].lower()

        if entry_text and entry_text in chunk_lower:
            pages.add(entry["page"])

    if not pages:
        return None, None

    return min(pages), max(pages)


# ============================================================
# EXTRACT MARKDOWN USING DOCLING
# ============================================================

def extract_markdown(source):

    converter = DocumentConverter()

    result = converter.convert(source)

    document = result.document

    markdown = document.export_to_markdown()

    return document, markdown


# ============================================================
# LANGCHAIN SPLIT
# ============================================================

def split_markdown(markdown):

    headers = [
        ("#", "topic"),
        ("##", "section"),
        ("###", "subsection"),
    ]

    splitter = MarkdownHeaderTextSplitter(
        headers_to_split_on=headers
    )

    docs = splitter.split_text(markdown)

    return docs


# ============================================================
# BUILD FINAL CHUNKS
# ============================================================

def build_chunks(docs, page_map):

    chunks = []
    index = 1

    for doc in docs:

        content = clean_chunk_text(doc.page_content)

        if not content or len(content) < 20:
            continue

        meta = doc.metadata

        topic = meta.get("topic")
        section = meta.get("section")
        subsection = meta.get("subsection")

        final_topic = subsection or section or topic or "General"

        pieces = rechunk_text(content)

        for piece in pieces:

            chunk = {

                "chunk_id": generate_chunk_id(index),
                "doc_type": DOC_TYPE,
                "doc_name": DOC_NAME,
                "source_url": SOURCE_URL,
                "Topic": final_topic,
                "num_tokens": token_count(piece),
                "content": piece

            }

            chunks.append(chunk)
            index += 1

    return chunks


# ============================================================
# MAIN PIPELINE
# ============================================================

def process():

    print("Extracting with Docling...")

    document, markdown = extract_markdown(SOURCE_URL)

    print("Promoting headings...")

    markdown = promote_caps_to_headers(markdown)

    print("Building page map...")

    page_map = build_page_map(document)

    print("Splitting document...")

    docs = split_markdown(markdown)

    print("Building chunks...")

    chunks = build_chunks(docs, page_map)

    print("Saving...")

    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        json.dump(chunks, f, indent=2, ensure_ascii=False)

    print(f"SUCCESS: {len(chunks)} chunks created")


# ============================================================
# RUN
# ============================================================

process()

Extracting with Docling...


[INFO] 2026-03-08 14:20:45,157 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-03-08 14:20:45,159 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-03-08 14:20:45,210 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_infer.pth
[INFO] 2026-03-08 14:20:45,211 [RapidOCR] main.py:50: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_infer.pth
[INFO] 2026-03-08 14:20:45,458 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-03-08 14:20:45,460 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-03-08 14:20:45,465 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_ptocr_mobile_v2.0_cls_infer.pth
[INFO] 2026-03-08 14:20:45,466 [RapidOCR] main.py:50: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_ptocr_mobile_v2.0_cls_infer.pth
[INFO] 2026-03-08 14:20:45,566 [RapidO

Promoting headings...
Building page map...
Splitting document...
Building chunks...
Saving...
SUCCESS: 28 chunks created


In [ ]:
import json

files = [
    "chunks.json",
    "coterminal-chunks.json",
    "final_chunks.json"
]

combined = []

for f in files:
    with open(f, "r", encoding="utf-8") as file:
        data = json.load(file)
        combined.extend(data)

with open("combined_chunks.json", "w", encoding="utf-8") as out:
    json.dump(combined, out, indent=2, ensure_ascii=False)

print("Total chunks:", len(combined))

Total chunks: 559
